In [0]:
%pip install -q databricks-openai databricks-ai-search
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI
from databricks.ai_search.client import AISearchClient

# Foundation model used for generation
MODEL = "system.ai.gemma-3-12b"

# Existing AI Search resources from Phase 5
AI_SEARCH_ENDPOINT = "eacc-rag-search"

INDEX_NAME = (
    "workspace.eacc_ecommerce_rag."
    "enterprise_knowledge_index"
)

# Retrieval configuration selected from Phase 5 evaluation
RETRIEVAL_QUERY_TYPE = "ann"
RETRIEVAL_TOP_K = 3

In [0]:
w = WorkspaceClient()

llm_client = DatabricksOpenAI(
    workspace_client=w,
    base_url=f"{w.config.host}/ai-gateway/mlflow/v1"
)

print("LLM client ready.")

In [0]:
search_client = AISearchClient(
    disable_notice=True
)

index = search_client.get_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

print("AI Search index connected.")

In [0]:
index_status = index.describe()

print(
    "Ready:",
    index_status["status"]["ready"]
)

print(
    "State:",
    index_status["status"]["detailed_state"]
)

In [0]:
def retrieve_context(
    question,
    top_k=RETRIEVAL_TOP_K
):
    results = index.similarity_search(
        query_text=question,

        columns=[
            "chunk_id",
            "document_title",
            "section_title",
            "source_text"
        ],

        num_results=top_k,
        query_type=RETRIEVAL_QUERY_TYPE
    )

    column_names = [
        column["name"]
        for column in results["manifest"]["columns"]
    ]

    rows = results["result"]["data_array"]

    retrieved = []

    for row in rows:
        values = row[:len(column_names)]

        retrieved.append(
            dict(zip(column_names, values))
        )

    return retrieved

In [0]:
retrieved = retrieve_context(
    "How many days does a customer have "
    "to report a damaged product?"
)

retrieved

In [0]:
def print_retrieved_context(retrieved):
    for rank, chunk in enumerate(
        retrieved,
        start=1
    ):
        print("=" * 80)
        print("RANK:", rank)
        print(
            "DOCUMENT:",
            chunk["document_title"]
        )
        print(
            "SECTION:",
            chunk["section_title"]
        )
        print(
            "CHUNK:",
            chunk["chunk_id"]
        )
        print()
        print(chunk["source_text"])
        print()

In [0]:
print_retrieved_context(retrieved)

In [0]:
def build_context(retrieved):
    context_blocks = []

    for i, chunk in enumerate(
        retrieved,
        start=1
    ):
        block = f"""
SOURCE {i}

Document: {chunk['document_title']}
Section: {chunk['section_title']}
Chunk ID: {chunk['chunk_id']}

{chunk['source_text']}
""".strip()

        context_blocks.append(block)

    return "\n\n---\n\n".join(
        context_blocks
    )

In [0]:
context = build_context(retrieved)

print(context)

In [0]:
RAG_SYSTEM_PROMPT = """
You are an enterprise knowledge assistant.

Answer questions using ONLY the supplied enterprise sources.

Rules:
- Treat the supplied sources as authoritative enterprise context.
- Do not invent company policies, thresholds, procedures, or facts.
- Do not use external knowledge to fill missing enterprise information.
- If the supplied sources do not contain enough information to answer,
  explicitly say that the answer cannot be determined from the
  retrieved enterprise sources.
- When abstaining because supporting evidence is absent,
  do NOT cite unrelated sources.
- Cite a source only when that source directly supports the claim.
- Distinguish policy requirements from operational recommendations.
- Do not treat operational thresholds as customer eligibility rules.
- Resolve similar numerical thresholds carefully.
- Keep the answer concise and business-focused.
- Cite supporting sources in the form [Source 1], [Source 2], etc.
"""

In [0]:
question = (
    "How many days after delivery does a customer "
    "have to report a damaged product?"
)

retrieved = retrieve_context(question)

context = build_context(retrieved)

rag_response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
ENTERPRISE SOURCES

{context}

QUESTION

{question}
"""
        }
    ],
    max_tokens=400
)

print(
    rag_response.choices[0].message.content
)

In [0]:
question = (
    "What percentage discount do employees receive "
    "when purchasing products from the company?"
)

retrieved = retrieve_context(question)

print_retrieved_context(retrieved)

In [0]:
context = build_context(retrieved)

In [0]:
insufficient_evidence_response = (
    llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": RAG_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
ENTERPRISE SOURCES

{context}

QUESTION

{question}
"""
            }
        ],
        max_tokens=400
    )
)

print(
    insufficient_evidence_response
    .choices[0]
    .message
    .content
)

In [0]:
conflicting_context = """
SOURCE 1

Document: Return and Refund Policy
Section: Standard Return Window

Customers may request a return within 30 calendar days
of the delivery date.

---

SOURCE 2

Document: Legacy Customer Returns Guide
Section: Standard Returns

Customers may request a standard return within 45 calendar days
of delivery.
"""

In [0]:
conflict_question = (
    "How many days does a customer have "
    "to make a standard return?"
)

conflict_response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT + """

Conflict handling:
- If authoritative retrieved sources provide conflicting values,
  do not silently choose one.
- Explicitly identify the conflict.
- State which values and sources disagree.
- Do not resolve the conflict unless the supplied sources provide
  a clear precedence rule.
"""
        },
        {
            "role": "user",
            "content": f"""
ENTERPRISE SOURCES

{conflicting_context}

QUESTION

{conflict_question}
"""
        }
    ],
    max_tokens=350
)

print(
    conflict_response.choices[0].message.content
)

In [0]:
governed_conflict_response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT + """

Source governance rules:

1. Documents marked as current enterprise policy take precedence
   over documents identified as legacy guidance.

2. If a current enterprise policy conflicts with legacy guidance,
   use the current enterprise policy and mention the conflicting
   legacy source.

3. Never combine conflicting numeric values.
"""
        },
        {
            "role": "user",
            "content": f"""
ENTERPRISE SOURCES

{conflicting_context}

QUESTION

{conflict_question}
"""
        }
    ],
    max_tokens=350
)

print(
    governed_conflict_response
    .choices[0]
    .message
    .content
)

In [0]:
def ask_rag(
    question,
    top_k=RETRIEVAL_TOP_K,
    show_sources=False
):
    retrieved = retrieve_context(
        question,
        top_k=top_k
    )

    context = build_context(retrieved)

    response = llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": RAG_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
ENTERPRISE SOURCES

{context}

QUESTION

{question}
"""
            }
        ],
        max_tokens=400
    )

    answer = response.choices[0].message.content

    if show_sources:
        print("QUESTION:")
        print(question)

        print("\nANSWER:")
        print(answer)

        print("\nRETRIEVED SOURCES:")
        print_retrieved_context(retrieved)

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved,
        "usage": response.usage
    }

In [0]:
test = ask_rag(
    "How long can a customer make a standard return?",
    show_sources=True
)

In [0]:
rag_confusion_tests = [
    "How long can a customer make a standard return?",

    "How many days after delivery does a customer have "
    "to report damaged or defective merchandise?",

    "At what monthly return rate does an individual "
    "product trigger standard review?",

    "At what monthly return rate does an individual "
    "product trigger priority investigation?",

    "At what monthly return rate does a product "
    "category trigger standard review?",

    "At what monthly return rate does a product "
    "category trigger priority review?",

    "At what monthly cancellation rate does standard "
    "operational review begin?",

    "At what monthly cancellation rate does priority "
    "review begin?"
]

In [0]:
rag_confusion_results = []

for question in rag_confusion_tests:
    result = ask_rag(question)

    rag_confusion_results.append(result)

    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(result["answer"])

    print()

In [0]:
for result in rag_confusion_results:
    print("=" * 100)
    print(result["question"])
    print()
    print_retrieved_context(
        result["sources"]
    )

In [0]:
import re

def extract_citations(answer):
    citation_numbers = []

    # Find bracketed blocks that contain the word Source.
    citation_blocks = re.findall(
        r"\[([^\]]*Source[^\]]*)\]",
        answer
    )

    for block in citation_blocks:
        numbers = re.findall(
            r"\d+",
            block
        )

        citation_numbers.extend(
            int(number)
            for number in numbers
        )

    return sorted(
        set(citation_numbers)
    )

In [0]:
tests = [
    "Answer [Source 1].",
    "Answer [Source 1, Source 2].",
    "Answer [Source 1, Source 2, Source 3].",
    "No citation here."
]

for test in tests:
    print(
        test,
        "->",
        extract_citations(test)
    )

In [0]:
example_answer = (
    "A damaged item must be reported within "
    "14 calendar days [Source 1, Source 2]."
)

extract_citations(example_answer)

In [0]:
def validate_citation_references(result):
    answer = result["answer"]
    sources = result["sources"]

    citations = extract_citations(answer)

    valid_source_numbers = set(
        range(1, len(sources) + 1)
    )

    invalid_citations = [
        citation
        for citation in citations
        if citation not in valid_source_numbers
    ]

    return {
        "citations": citations,
        "num_sources": len(sources),
        "invalid_citations": invalid_citations,
        "all_citations_valid": (
            len(invalid_citations) == 0
        )
    }

In [0]:
citation_check = validate_citation_references(
    rag_confusion_results[0]
)

citation_check

In [0]:
for result in rag_confusion_results:
    check = validate_citation_references(
        result
    )

    print("=" * 100)
    print("QUESTION:")
    print(result["question"])

    print("\nANSWER:")
    print(result["answer"])

    print("\nCITATIONS:")
    print(check["citations"])

    print(
        "ALL REFERENCES VALID:",
        check["all_citations_valid"]
    )

In [0]:
def show_cited_sources(result):
    citations = extract_citations(
        result["answer"]
    )

    for citation in citations:
        source = result["sources"][
            citation - 1
        ]

        print("=" * 80)
        print("SOURCE:", citation)
        print(
            "DOCUMENT:",
            source["document_title"]
        )
        print(
            "SECTION:",
            source["section_title"]
        )
        print()
        print(source["source_text"])
        print()

In [0]:
print(
    rag_confusion_results[0]["answer"]
)

show_cited_sources(
    rag_confusion_results[0]
)

In [0]:
abstention_result = ask_rag(
    "What percentage discount do employees receive "
    "when purchasing products from the company?"
)

print(
    abstention_result["answer"]
)

print(
    "Citations:",
    extract_citations(
        abstention_result["answer"]
    )
)

In [0]:
multi_source_test = ask_rag(
    "Explain the difference between the standard "
    "return window and the damaged-product reporting window.",
    show_sources=True
)

In [0]:
validate_citation_references(
    multi_source_test
)

In [0]:
show_cited_sources(
    multi_source_test
)

In [0]:
RAG_RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "grounded_rag_response",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "status": {
                    "type": "string",
                    "enum": [
                        "ANSWERED",
                        "INSUFFICIENT_EVIDENCE",
                        "CONFLICTING_EVIDENCE"
                    ]
                },
                "answer": {
                    "type": "string"
                },
                "cited_source_numbers": {
                    "type": "array",
                    "items": {
                        "type": "integer"
                    }
                }
            },
            "required": [
                "status",
                "answer",
                "cited_source_numbers"
            ],
            "additionalProperties": False
        }
    }
}

In [0]:
import json

def ask_rag_structured(
    question,
    top_k=RETRIEVAL_TOP_K
):
    retrieved = retrieve_context(
        question,
        top_k=top_k
    )

    context = build_context(
        retrieved
    )

    response = llm_client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
You are an enterprise knowledge assistant.

Use ONLY the supplied enterprise sources.

Rules:

- If the sources directly support an answer:
  status = ANSWERED

- If the sources do not contain enough evidence:
  status = INSUFFICIENT_EVIDENCE
  and cited_source_numbers MUST be []

- If authoritative sources contain unresolved conflicting facts:
  status = CONFLICTING_EVIDENCE

- Cite only sources that directly support the answer.
- Never cite a source merely because it was retrieved.
- Source numbers correspond to SOURCE 1, SOURCE 2, etc.
- Do not invent enterprise facts.
- Preserve numerical thresholds and boundary language exactly.
"""
            },

            {
                "role": "user",
                "content": f"""
ENTERPRISE SOURCES

{context}

QUESTION

{question}
"""
            }
        ],

        response_format=RAG_RESPONSE_SCHEMA,
        max_tokens=400
    )

    parsed = json.loads(
        response.choices[0].message.content
    )

    return {
        "question": question,
        "status": parsed["status"],
        "answer": parsed["answer"],
        "citations": parsed["cited_source_numbers"],
        "sources": retrieved,
        "usage": response.usage
    }

In [0]:
def render_rag_answer(result):
    answer = result["answer"]
    citations = result["citations"]

    if citations:
        citation_text = " ".join(
            f"[Source {number}]"
            for number in citations
        )

        return f"{answer} {citation_text}"

    return answer

In [0]:
good_test = ask_rag_structured(
    "How many days after delivery does a customer "
    "have to report damaged merchandise?"
)

print(good_test)

print()
print(
    render_rag_answer(good_test)
)

In [0]:
missing_test = ask_rag_structured(
    "What percentage discount do employees receive "
    "when purchasing company products?"
)

print(missing_test)

print()
print(
    render_rag_answer(missing_test)
)

In [0]:
rag_eval = [
    {
        "question": "How long can a customer make a standard return?",
        "expected_status": "ANSWERED",
        "expected_terms": ["30", "calendar days"],
        "should_cite": True
    },

    {
        "question": "How many days after delivery must damaged merchandise be reported?",
        "expected_status": "ANSWERED",
        "expected_terms": ["14", "calendar days"],
        "should_cite": True
    },

    {
        "question": "At what monthly return rate does an individual product trigger standard review?",
        "expected_status": "ANSWERED",
        "expected_terms": ["10%", "exceed"],
        "should_cite": True
    },

    {
        "question": "At what monthly return rate does an individual product trigger priority investigation?",
        "expected_status": "ANSWERED",
        "expected_terms": ["18%", "exceed"],
        "should_cite": True
    },

    {
        "question": "At what monthly return rate does a product category trigger standard review?",
        "expected_status": "ANSWERED",
        "expected_terms": ["12%", "exceed"],
        "should_cite": True
    },

    {
        "question": "At what monthly return rate does a product category trigger priority review?",
        "expected_status": "ANSWERED",
        "expected_terms": ["15%", "exceed"],
        "should_cite": True
    },

    {
        "question": "At what monthly cancellation rate does priority review begin?",
        "expected_status": "ANSWERED",
        "expected_terms": ["15%", "above"],
        "should_cite": True
    },

    {
        "question": "What percentage discount do employees receive when purchasing company products?",
        "expected_status": "INSUFFICIENT_EVIDENCE",
        "expected_terms": [],
        "should_cite": False
    },

    {
        "question": "What exactly does net revenue mean in our analytics model?",
        "expected_status": "ANSWERED",
        "expected_terms": [
            "cancellation",
            "return",
            "adjust"
        ],
        "should_cite": True
    },

    {
        "question": "Can a shipped order normally be handled as a standard cancellation?",
        "expected_status": "ANSWERED",
        "expected_terms": [
            "not",
            "standard cancellation"
        ],
        "should_cite": True
    }
]

In [0]:
def evaluate_rag_case(test):
    result = ask_rag_structured(
        test["question"]
    )

    answer_lower = result["answer"].lower()

    status_correct = (
        result["status"]
        == test["expected_status"]
    )

    terms_correct = all(
        term.lower() in answer_lower
        for term in test["expected_terms"]
    )

    if test["should_cite"]:
        citation_behavior_correct = (
            len(result["citations"]) > 0
        )
    else:
        citation_behavior_correct = (
            len(result["citations"]) == 0
        )

    valid_source_numbers = set(
        range(1, len(result["sources"]) + 1)
    )

    citation_references_valid = all(
        citation in valid_source_numbers
        for citation in result["citations"]
    )

    passed = all([
        status_correct,
        terms_correct,
        citation_behavior_correct,
        citation_references_valid
    ])

    return {
        "question": test["question"],
        "status": result["status"],
        "answer": result["answer"],
        "citations": result["citations"],

        "status_correct": status_correct,
        "terms_correct": terms_correct,
        "citation_behavior_correct":
            citation_behavior_correct,

        "citation_references_valid":
            citation_references_valid,

        "passed": passed,

        "usage": result["usage"]
    }

In [0]:
rag_eval_results = []

for test in rag_eval:
    result = evaluate_rag_case(test)

    rag_eval_results.append(result)

    print("=" * 100)
    print("QUESTION:")
    print(result["question"])

    print("\nSTATUS:")
    print(result["status"])

    print("\nANSWER:")
    print(result["answer"])

    print("\nCITATIONS:")
    print(result["citations"])

    print("\nPASS:", result["passed"])

In [0]:
total = len(rag_eval_results)

passed = sum(
    1
    for result in rag_eval_results
    if result["passed"]
)

status_correct = sum(
    1
    for result in rag_eval_results
    if result["status_correct"]
)

terms_correct = sum(
    1
    for result in rag_eval_results
    if result["terms_correct"]
)

citation_behavior_correct = sum(
    1
    for result in rag_eval_results
    if result["citation_behavior_correct"]
)

citation_refs_valid = sum(
    1
    for result in rag_eval_results
    if result["citation_references_valid"]
)

print(
    f"Overall pass rate: "
    f"{passed / total:.2%}"
)

print(
    f"Status accuracy: "
    f"{status_correct / total:.2%}"
)

print(
    f"Answer-content accuracy: "
    f"{terms_correct / total:.2%}"
)

print(
    f"Citation behavior accuracy: "
    f"{citation_behavior_correct / total:.2%}"
)

print(
    f"Citation-reference validity: "
    f"{citation_refs_valid / total:.2%}"
)

In [0]:
total_tokens = sum(
    result["usage"].total_tokens
    for result in rag_eval_results
)

avg_tokens = (
    total_tokens / len(rag_eval_results)
)

print(
    "Total evaluation tokens:",
    total_tokens
)

print(
    "Average tokens per RAG request:",
    round(avg_tokens, 2)
)

In [0]:
def enterprise_knowledge_search(question):
    result = ask_rag_structured(question)

    return {
        "status": result["status"],
        "answer": render_rag_answer(result),

        "sources": [
            {
                "source_number": index + 1,
                "document": source["document_title"],
                "section": source["section_title"],
                "chunk_id": source["chunk_id"]
            }
            for index, source
            in enumerate(result["sources"])
            if index + 1 in result["citations"]
        ]
    }

In [0]:
enterprise_knowledge_search(
    "When does a product category require priority review?"
)

In [0]:
def enterprise_knowledge_search(question):
    result = ask_rag_structured(question)

    cited_sources = []

    for source_number in result["citations"]:
        source = result["sources"][source_number - 1]

        cited_sources.append({
            "source_number": source_number,
            "document": source["document_title"],
            "section": source["section_title"],
            "chunk_id": source["chunk_id"]
        })

    return {
        "status": result["status"],
        "answer": render_rag_answer(result),
        "sources": cited_sources
    }

In [0]:
result = enterprise_knowledge_search(
    "When does a product category require priority review?"
)

result

In [0]:
result = enterprise_knowledge_search(
    "What percentage discount do employees receive?"
)

result